In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
SEED = 42
np.random.seed(SEED)

In [ ]:
# 加载完整训练数据
dtypes = {
    'stock_id': 'int16', 'date_id': 'int16', 'seconds_in_bucket': 'int16',
    'imbalance_size': 'float32', 'imbalance_buy_sell_flag': 'int8',
    'reference_price': 'float32', 'matched_size': 'float32',
    'far_price': 'float32', 'near_price': 'float32',
    'bid_price': 'float32', 'bid_size': 'float32',
    'ask_price': 'float32', 'ask_size': 'float32', 'wap': 'float32',
    'target': 'float32'
}
train = pd.read_csv('/kaggle/input/datasets/christophergarsia/dataset-01/train.csv', dtype=dtypes)
print(f'原始数据形状: {train.shape}')

# 时间切分：取前 80% 日期作为训练，后 20% 作为验证
date_ids = sorted(train['date_id'].unique())
split_idx = int(len(date_ids) * 0.8)
train_dates = date_ids[:split_idx]
val_dates = date_ids[split_idx:]
print(f'训练日期: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)} 天)')
print(f'验证日期: {val_dates[0]} ~ {val_dates[-1]} ({len(val_dates)} 天)')

train_data = train[train['date_id'].isin(train_dates)].copy()
val_data   = train[train['date_id'].isin(val_dates)].copy()
del train

print(f'训练集: {train_data.shape}, 验证集: {val_data.shape}')

In [ ]:
def build_champion_features(df):
    """
    根据冠军方案精髓构建特征集
    """
    data = df.copy()
    
    # ==================== 基础量 ====================
    data['mid_price'] = (data['bid_price'] + data['ask_price']) / 2
    data['spread'] = data['ask_price'] - data['bid_price']
    data['book_imbalance'] = (data['bid_size'] - data['ask_size']) / (data['bid_size'] + data['ask_size'] + 1e-6)
    data['wap_deviation'] = data['wap'] - data['mid_price']
    data['auction_spread'] = data['far_price'] - data['near_price']
    data['ref_price_deviation'] = (data['mid_price'] - data['reference_price']) / (data['reference_price'] + 1e-6)
    
    # ==================== 动态股票内归一化 ====================
    data = data.sort_values(['stock_id', 'date_id', 'seconds_in_bucket'])
    
    def rolling_zscore(series, window=200):
        """滚动 z-score 归一化"""
        mean = series.rolling(window, min_periods=1).mean()
        std  = series.rolling(window, min_periods=1).std()
        return (series - mean) / (std + 1e-6)
    
    # 对核心信号做滚动归一化（按股票分组）
    data['book_imbalance_norm'] = data.groupby('stock_id')['book_imbalance'].transform(rolling_zscore)
    data['imbalance_size_norm'] = data.groupby('stock_id')['imbalance_size'].transform(rolling_zscore)
    data['wap_deviation_norm'] = data.groupby('stock_id')['wap_deviation'].transform(rolling_zscore)
    
    # ==================== 动量与加速度 ====================
    data['imb_velocity'] = data.groupby('stock_id')['book_imbalance'].diff()
    data['imb_acceleration'] = data.groupby('stock_id')['imb_velocity'].diff()
    data['imbalance_velocity'] = data.groupby('stock_id')['imbalance_size'].diff()
    
    # ==================== 多时间窗口统计 ====================
    def add_window_stats(group):
        group = group.sort_values('seconds_in_bucket')
        for w in [5, 10, 30]:
            # 不平衡动量的滚动均值
            group[f'imb_vel_mean_{w}'] = group['imb_velocity'].rolling(w, min_periods=1).mean()
            # 不平衡加速度的滚动标准差
            group[f'imb_acc_std_{w}'] = group['imb_acceleration'].rolling(w, min_periods=1).std()
            # 价差滚动均值
            group[f'spread_mean_{w}'] = group['spread'].rolling(w, min_periods=1).mean()
            # 价格波动率
            group[f'price_vol_{w}'] = group['mid_price'].pct_change().rolling(w, min_periods=1).std()
        return group
    
    data = data.groupby('stock_id', group_keys=False).apply(add_window_stats)
    
    # ==================== 当前值与均值的偏离 ====================
    data['imb_vs_mean_5'] = data['book_imbalance'] - data.groupby('stock_id')['book_imbalance'].transform(
        lambda x: x.rolling(5, min_periods=1).mean())
    data['imb_vs_mean_30'] = data['book_imbalance'] - data.groupby('stock_id')['book_imbalance'].transform(
        lambda x: x.rolling(30, min_periods=1).mean())
    
    # ==================== 拍卖簿与订单簿背离 ====================
    data['auction_book_divergence'] = (
        (data['imbalance_size_norm'] - data['book_imbalance_norm']) /
        (np.abs(data['imbalance_size_norm']) + np.abs(data['book_imbalance_norm']) + 1e-6)
    )
    
    # 填充 NaN
    data = data.fillna(0)
    return data

print('开始构建冠军方案特征...')
train_features = build_champion_features(train_data)
val_features   = build_champion_features(val_data)
print('特征工程完成！')
print(f'训练特征形状: {train_features.shape}')

In [ ]:
feature_cols = [
    'seconds_in_bucket',
    'mid_price', 'spread', 'book_imbalance', 'wap_deviation',
    'auction_spread', 'ref_price_deviation',
    'book_imbalance_norm', 'imbalance_size_norm', 'wap_deviation_norm',
    'imb_velocity', 'imb_acceleration', 'imbalance_velocity',
    'imb_vel_mean_5', 'imb_vel_mean_10', 'imb_vel_mean_30',
    'imb_acc_std_5', 'imb_acc_std_10', 'imb_acc_std_30',
    'spread_mean_5', 'spread_mean_10', 'spread_mean_30',
    'price_vol_5', 'price_vol_10', 'price_vol_30',
    'imb_vs_mean_5', 'imb_vs_mean_30',
    'auction_book_divergence'
]

# 确保所有特征都在数据中
feature_cols = [c for c in feature_cols if c in train_features.columns]
print(f'使用特征数量: {len(feature_cols)}')

X_train = train_features[feature_cols]
y_train = train_features['target']
X_val   = val_features[feature_cols]
y_val   = val_features['target']

In [ ]:
# 基于训练集计算每只股票的 target 均值和标准差
stock_stats = y_train.groupby(train_features['stock_id']).agg(['mean', 'std'])
stock_stats.columns = ['target_mean', 'target_std']

# 映射到训练和验证集
train_features['target_mean'] = train_features['stock_id'].map(stock_stats['target_mean'].to_dict())
train_features['target_std']  = train_features['stock_id'].map(stock_stats['target_std'].to_dict())
val_features['target_mean']   = val_features['stock_id'].map(stock_stats['target_mean'].to_dict())
val_features['target_std']    = val_features['stock_id'].map(stock_stats['target_std'].to_dict())

# 标准化
eps = 1e-6
y_train_std = (y_train - train_features['target_mean']) / (train_features['target_std'] + eps)
y_val_std   = (y_val - val_features['target_mean']) / (val_features['target_std'] + eps)

print('目标标准化完成')

In [ ]:
params = {
    'boosting_type': 'gbdt',
    'objective': 'mae',
    'metric': 'mae',
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 31,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'bagging_freq': 1,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': 100
}

model = lgb.LGBMRegressor(
    n_estimators=5000,
    **params
)

# 早停回调
callbacks = [lgb.early_stopping(50), lgb.log_evaluation(100)]

model.fit(
    X_train, y_train_std,
    eval_set=[(X_val, y_val_std)],
    callbacks=callbacks
)

print('训练完成')

In [ ]:
# 预测（标准化输出）
y_pred_std = model.predict(X_val)

# 逆变换为原始 target
y_pred = y_pred_std * val_features['target_std'] + val_features['target_mean']

# MAE
mae = np.mean(np.abs(y_pred - y_val))
print(f'验证集 MAE: {mae:.6f}')

# 基线（历史均值）
baseline_pred = val_features['target_mean']  # 因为标准化时均值就是历史均值
baseline_mae = np.mean(np.abs(baseline_pred - y_val))
print(f'基线 MAE:   {baseline_mae:.6f}')
print(f'提升幅度:   {(1 - mae/baseline_mae)*100:.2f}%')

# 特征重要性
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 15 特征重要性:')
print(importance.head(15).to_string(index=False))

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. 获取训练集预测（标准化尺度）
y_train_pred_std = model.predict(X_train)

# 2. 还原为原始 target 尺度
y_train_pred = y_train_pred_std * train_features['target_std'] + train_features['target_mean']

# 3. 拟合缩放器：真实值 ~ 预测值
scaler = LinearRegression()
scaler.fit(y_train_pred.values.reshape(-1, 1), y_train.values)

# 4. 对验证集预测进行缩放
y_val_pred_scaled = scaler.predict(y_pred.values.reshape(-1, 1))

# 5. 评估缩放后的 MAE
mae_scaled = np.mean(np.abs(y_val_pred_scaled - y_val))
print(f'缩放后验证集 MAE: {mae_scaled:.6f}')
print(f'缩放前 MAE: {mae:.6f}')
print(f'缩放提升: {mae - mae_scaled:.6f}')

# 打印缩放系数
print(f'\n缩放器系数: {scaler.coef_[0]:.6f}')
print(f'缩放器截距: {scaler.intercept_:.6f}')